In [13]:
import json

train_path = "query/train_data.json"
test_path = "query/test_data.json"
val_path = "query/val_data.json"
with open(train_path, "r") as f:
    train_data = json.load(f)
with open(test_path, "r") as f:
    test_data = json.load(f)
with open(val_path, "r") as f:
    val_data = json.load(f)
print("✅ Total queries in train_data.json:", len(train_data))
print("✅ Total queries in test_data.json:", len(test_data))
print("✅ Total queries in val_data.json:", len(val_data))
video_ids = set(item["video"] for item in train_data)
print("✅ Unique videos in train:", len(video_ids))
video_ids = set(item["video"] for item in test_data)
print("✅ Unique videos in test:", len(video_ids))
video_ids = set(item["video"] for item in val_data)
print("✅ Unique videos in val:", len(video_ids))

✅ Total queries in train_data.json: 33005
✅ Total queries in test_data.json: 4021
✅ Total queries in val_data.json: 4180
✅ Unique videos in train: 8511
✅ Unique videos in test: 1037
✅ Unique videos in val: 1094


In [1]:
import json

def load_video_ids(path):
    with open(path, "r") as f:
        data = json.load(f)
    return set(item["video"] for item in data)

train_vids = load_video_ids("dataset/query/train_data.json")
val_vids   = load_video_ids("dataset/query/val_data.json")
test_vids  = load_video_ids("dataset/query/test_data.json")

all_query_vids = train_vids | val_vids | test_vids

print("Train unique videos:", len(train_vids))
print("Val unique videos:", len(val_vids))
print("Test unique videos:", len(test_vids))
print("Total unique query videos:", len(all_query_vids))
import h5py

rgb_path = "dataset/features/average_rgb_feats.h5"

with h5py.File(rgb_path, "r") as f:
    rgb_vids = set(f.keys())

print("RGB feature videos:", len(rgb_vids))

flow_path = "dataset/features/average_flow_feats.h5"

with h5py.File(flow_path, "r") as f:
    flow_vids = set(f.keys())

print("Flow feature videos:", len(flow_vids))

gflow_path = "dataset/features/average_global_flow.h5"

with h5py.File(gflow_path, "r") as f:
    gflow_vids = set(f.keys())

print("Global flow videos:", len(gflow_vids))

print("\n🔍 MATCHING ANALYSIS")

# Query vs RGB
missing_rgb = all_query_vids - rgb_vids
print("Missing in RGB:", len(missing_rgb))

# Query vs FLOW
missing_flow = all_query_vids - flow_vids
print("Missing in Flow:", len(missing_flow))

# RGB vs FLOW consistency
rgb_flow_diff = rgb_vids ^ flow_vids
print("RGB vs Flow mismatch:", len(rgb_flow_diff))

# Final usable videos
valid_vids = all_query_vids & rgb_vids & flow_vids
print("\n✅ Fully usable videos:", len(valid_vids))

print("\nSample missing in flow:", list(missing_flow)[:5])
print("Sample missing in rgb:", list(missing_rgb)[:5])

Train unique videos: 8511
Val unique videos: 1094
Test unique videos: 1037
Total unique query videos: 10642
RGB feature videos: 10642
Flow feature videos: 13803
Global flow videos: 10642

🔍 MATCHING ANALYSIS
Missing in RGB: 0
Missing in Flow: 0
RGB vs Flow mismatch: 3161

✅ Fully usable videos: 10642

Sample missing in flow: []
Sample missing in rgb: []


In [3]:
import json
import os
import h5py


# ---------------- NORMALIZATION FUNCTION ----------------
def normalize_vid(vid):
    """
    Convert:
    train/abc.mp4 → abc
    abc.avi       → abc
    """
    vid = os.path.basename(vid)          # remove path
    vid = os.path.splitext(vid)[0]       # remove extension
    return vid


# ---------------- LOAD JSON ----------------
def load_json_vids(path):
    with open(path, "r") as f:
        data = json.load(f)
    return set(normalize_vid(item["video"]) for item in data)


# ---------------- LOAD JSONL ----------------
def load_jsonl_vids(path):
    vids = set()
    with open(path, "r") as f:
        for line in f:
            if line.strip():
                item = json.loads(line)
                vids.add(normalize_vid(item["video"]))
    return vids


# ---------------- LOAD H5 ----------------
def load_h5_vids(path):
    with h5py.File(path, "r") as f:
        return set(normalize_vid(k) for k in f.keys())


# ---------------- PATHS ----------------
train_json = "dataset/query/train_data.json"
val_json   = "dataset/query/val_data.json"
test_json  = "dataset/query/test_data.json"

train_jsonl = "dataset/query/train.jsonl"
test_jsonl  = "dataset/query/test.jsonl"

rgb_h5  = "dataset/features/average_rgb_feats.h5"
flow_h5 = "dataset/features/average_flow_feats.h5"


# ---------------- LOAD ----------------
print("\n🚀 Loading all datasets...")

train_vids = load_json_vids(train_json)
val_vids   = load_json_vids(val_json)
test_vids  = load_json_vids(test_json)

train_jsonl_vids = load_jsonl_vids(train_jsonl)
test_jsonl_vids  = load_jsonl_vids(test_jsonl)

rgb_vids  = load_h5_vids(rgb_h5)
flow_vids = load_h5_vids(flow_h5)


# ---------------- COMBINE ----------------
all_query_vids = train_vids | val_vids | test_vids


# ---------------- CHECK JSON vs JSONL ----------------
print("\n================ JSON vs JSONL =================")

train_common = train_vids & train_jsonl_vids
test_common  = test_vids & test_jsonl_vids

print("Train common videos:", len(train_common))
print("Test common videos:", len(test_common))

print("Train missing:", len(train_vids - train_jsonl_vids))
print("Test missing:", len(test_vids - test_jsonl_vids))


# ---------------- CHECK QUERY vs FEATURES ----------------
print("\n================ QUERY vs FEATURES =================")

common_rgb  = all_query_vids & rgb_vids
common_flow = all_query_vids & flow_vids

print("Common with RGB:", len(common_rgb))
print("Common with Flow:", len(common_flow))

print("Missing in RGB:", len(all_query_vids - rgb_vids))
print("Missing in Flow:", len(all_query_vids - flow_vids))


# ---------------- FINAL COMMON SET ----------------
print("\n================ FINAL COMMON VIDEOS =================")

final_common = all_query_vids & rgb_vids & flow_vids

print("✅ Final usable videos:", len(final_common))


# ---------------- SAMPLE CHECK ----------------
print("\nSample common videos:", list(final_common)[:5])


🚀 Loading all datasets...

================ JSON vs JSONL =================
Train common videos: 8395
Test common videos: 1004
Train missing: 0
Test missing: 0

================ QUERY vs FEATURES =================
Common with RGB: 10464
Common with Flow: 10464
Missing in RGB: 0
Missing in Flow: 0

================ FINAL COMMON VIDEOS =================
✅ Final usable videos: 10464

Sample common videos: ['14914765@N00_3646338941_6f7b21792c', '16870059@N04_3082013512_9739ffbf56', '14284621@N06_7326456132_82ecae939b', '45213160@N00_7127255435_0672979800', '14193725@N00_4526643441_f784ef2e45']


In [4]:
import json
import os
import h5py


# ================= NORMALIZATION =================
def normalize_vid(vid):
    vid = os.path.basename(vid)
    vid = os.path.splitext(vid)[0]
    return vid


# ================= LOAD JSON =================
def load_json_vids(path):
    with open(path, "r") as f:
        data = json.load(f)
    return set(normalize_vid(item["video"]) for item in data)


# ================= LOAD JSONL =================
def load_jsonl_vids(path):
    vids = set()
    with open(path, "r") as f:
        for line in f:
            if line.strip():
                item = json.loads(line)
                vids.add(normalize_vid(item["video"]))
    return vids


# ================= LOAD H5 =================
def load_h5_vids(path):
    with h5py.File(path, "r") as f:
        return set(normalize_vid(k) for k in f.keys())


# ================= PATHS =================
train_json = "dataset/query/train_data.json"
val_json   = "dataset/query/val_data.json"
test_json  = "dataset/query/test_data.json"

train_jsonl = "dataset/query/train.jsonl"
test_jsonl  = "dataset/query/test.jsonl"

rgb_h5  = "dataset/features/average_rgb_feats.h5"
flow_h5 = "dataset/features/average_flow_feats.h5"


# ================= LOAD ALL =================
print("\n🚀 Loading all sources...\n")

train_vids = load_json_vids(train_json)
val_vids   = load_json_vids(val_json)
test_vids  = load_json_vids(test_json)

json_vids = train_vids | val_vids | test_vids

train_jsonl_vids = load_jsonl_vids(train_jsonl)
test_jsonl_vids  = load_jsonl_vids(test_jsonl)

jsonl_vids = train_jsonl_vids | test_jsonl_vids

rgb_vids  = load_h5_vids(rgb_h5)
flow_vids = load_h5_vids(flow_h5)


# ================= BASIC COUNTS =================
print("JSON total videos:", len(json_vids))
print("JSONL total videos:", len(jsonl_vids))
print("RGB videos:", len(rgb_vids))
print("Flow videos:", len(flow_vids))


# ================= FEATURE INTERSECTION =================
feature_vids = rgb_vids & flow_vids
print("\nRGB ∩ FLOW:", len(feature_vids))


# ================= FINAL COMMON =================
common_all = json_vids & jsonl_vids & feature_vids

print("\n================ FINAL RESULT =================")
print("✅ Common across ALL sources:", len(common_all))


# ================= MISSING ANALYSIS =================
print("\n================ MISSING =================")

print("Missing in JSONL:", len(json_vids - jsonl_vids))
print("Missing in JSON:", len(jsonl_vids - json_vids))

print("Missing in RGB:", len(json_vids - rgb_vids))
print("Missing in FLOW:", len(json_vids - flow_vids))


# ================= SAMPLE CHECK =================
print("\nSample common videos:", list(common_all)[:5])
print("Sample missing in RGB:", list((json_vids - rgb_vids))[:5])
print("Sample missing in JSONL:", list((json_vids - jsonl_vids))[:5])


🚀 Loading all sources...

JSON total videos: 10464
JSONL total videos: 9399
RGB videos: 10464
Flow videos: 13557

RGB ∩ FLOW: 10464

================ FINAL RESULT =================
✅ Common across ALL sources: 9399

================ MISSING =================
Missing in JSONL: 1065
Missing in JSON: 0
Missing in RGB: 0
Missing in FLOW: 0

Sample common videos: ['14914765@N00_3646338941_6f7b21792c', '16870059@N04_3082013512_9739ffbf56', '14284621@N06_7326456132_82ecae939b', '14193725@N00_4526643441_f784ef2e45', '55626021@N03_5901326672_feb25afa4b']
Sample missing in RGB: []
Sample missing in JSONL: ['55231259@N00_5620377958_132cabfe38', '64735396@N03_9558743853_cf28267460', '28214861@N00_6946656970_d34739e7cf', '45213160@N00_7127255435_0672979800', '24311648@N00_3081934824_e7cb9365ae']
